# Research: ML-XGBoost Strategy

## Contexte
- **Performance actuelle (v2)** : Sharpe 0.566, CAGR 14.8%, MaxDD 38.6%
- **Univers** : 15 actions liquides (AAPL, MSFT, GOOGL, AMZN, NVDA, META, TSLA, JPM, V, WMT, DIS, NFLX, PYPL, ADBE, CRM)
- **Modele** : GradientBoostingRegressor (100 arbres, depth 5, lr 0.03, subsample 0.8)
- **Features** : 24 indicateurs techniques (returns, RSI, BB, MACD, Stochastique, ATR, momentum, volatilite, volume, ratios SMA)
- **Rebalancement** : Bihebdomadaire (lundis alternants : entrainement / trading)
- **Seuil** : 0.001 (rendement 10j predit > 0.1%)
- **Positions max** : 9, allocation ~10% chacune (90% total)

## Hypotheses a tester

| # | Hypothese | Rationale |
|---|-----------|----------|
| H1 | Learning rate (0.01, 0.03, 0.05, 0.10) | LR bas = convergence lente mais meilleure generalisation. LR haut = rapide mais surapprentissage |
| H2 | N estimateurs (50, 100, 200, 300) | Plus d'arbres = meilleure approximation mais risque d'overfitting et cout |
| H3 | Seuil de prediction (0.0, 0.001, 0.005, 0.01) | Seuil bas = plus de positions. Seuil haut = selectivite accrue |
| H4 | Max positions (3, 5, 7, 9, 12) | Concentration vs diversification. 9 positions actuel est deja large |
| H5 | Subsample ratio (0.6, 0.7, 0.8, 1.0) | Subsampling reduit la variance et ameliore la generalisation |

## Methodologie
- **Donnees** : yfinance, 2015-01-01 a 2026-01-01 (~11 ans)
- **Backtest** : Standalone (pandas), fenetre glissante train/test
- **Metriques** : Sharpe, CAGR, MaxDD, volatilite annualisee
- **Benchmark** : SPY buy-and-hold

> **[REFERENCE QC Cloud]**
> Ce notebook utilise des modeles ML lourds (RandomForest/XGBoost/VIX) necessitant des donnees QC Cloud.
> Pour executer : https://www.quantconnect.com/research


## 1. Chargement des donnees et utilitaires

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

TICKERS_15 = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA',
              'JPM', 'V', 'WMT', 'DIS', 'NFLX', 'PYPL', 'ADBE', 'CRM']
BENCHMARK = 'SPY'

print("Downloading price data...")
all_tickers = list(set(TICKERS_15 + [BENCHMARK]))
raw = yf.download(all_tickers, start='2015-01-01', end='2026-01-01', auto_adjust=True)
prices = raw['Close'].dropna()
print(f"Data range: {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"Trading days: {len(prices)}")
print(f"Tickers loaded: {list(prices.columns)}")

### Feature engineering

Reproduction exacte des 24 features utilises dans `main.py`.

In [ ]:
def calculate_features(df):
    """Calculate 24 technical features matching main.py."""
    closes = df['close']
    volumes = df['volume']
    highs = df['high']
    lows = df['low']
    returns = closes.pct_change()

    # SMA
    sma_5 = closes.rolling(5).mean()
    sma_10 = closes.rolling(10).mean()
    sma_20 = closes.rolling(20).mean()
    sma_50 = closes.rolling(50).mean()

    ema_12 = closes.ewm(span=12).mean()
    ema_26 = closes.ewm(span=26).mean()

    # RSI
    delta = closes.diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))

    # Bollinger Bands
    bb_middle = closes.rolling(20).mean()
    bb_std = closes.rolling(20).std()
    bb_upper = bb_middle + 2 * bb_std
    bb_lower = bb_middle - 2 * bb_std
    bb_width = (bb_upper - bb_lower) / bb_middle
    bb_position = (closes - bb_lower) / (bb_upper - bb_lower)

    # MACD
    macd = ema_12 - ema_26
    macd_signal = macd.ewm(span=9).mean()
    macd_histogram = macd - macd_signal

    # Stochastic
    rolling_min = lows.rolling(14).min()
    rolling_max = highs.rolling(14).max()
    denom = rolling_max - rolling_min
    denom = denom.replace(0, np.nan)
    stoch_k = 100 * (closes - rolling_min) / denom
    stoch_d = stoch_k.rolling(3).mean()

    # ATR
    high_low = highs - lows
    high_close = np.abs(highs - closes.shift())
    low_close = np.abs(lows - closes.shift())
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    atr = true_range.rolling(14).mean()

    # Momentum
    momentum_5 = closes / closes.shift(5) - 1
    momentum_10 = closes / closes.shift(10) - 1
    momentum_20 = closes / closes.shift(20) - 1

    # Volatility
    volatility_5 = returns.rolling(5).std()
    volatility_20 = returns.rolling(20).std()

    # Volume
    volume_sma = volumes.rolling(20).mean()
    volume_ratio = volumes / volume_sma
    volume_change = volumes.pct_change()

    # Price ratios
    price_to_sma5 = closes / sma_5
    price_to_sma20 = closes / sma_20
    price_to_sma50 = closes / sma_50

    features = pd.DataFrame({
        'returns': returns,
        'rsi': rsi,
        'bb_width': bb_width,
        'bb_position': bb_position,
        'macd': macd,
        'macd_signal': macd_signal,
        'macd_hist': macd_histogram,
        'stoch_k': stoch_k,
        'stoch_d': stoch_d,
        'atr': atr,
        'atr_ratio': atr / closes,
        'mom_5': momentum_5,
        'mom_10': momentum_10,
        'mom_20': momentum_20,
        'vol_5': volatility_5,
        'vol_20': volatility_20,
        'volume_ratio': volume_ratio,
        'volume_change': volume_change,
        'price_sma5': price_to_sma5,
        'price_sma20': price_to_sma20,
        'price_sma50': price_to_sma50,
        'sma_ratio_5_20': sma_5 / sma_20,
        'sma_ratio_10_50': sma_10 / sma_50,
    })

    return features.fillna(0).replace([np.inf, -np.inf], 0)

print("Feature engineering function defined (24 features).")

### Moteur de backtest XGBoost

In [ ]:
def backtest_xgb_strategy(prices, tickers, n_estimators=100, max_depth=5,
                          learning_rate=0.03, subsample=0.8,
                          threshold=0.001, max_positions=9,
                          train_months=6, rebal_weeks=2,
                          name='XGB Strategy'):
    """Backtest GradientBoosting strategy with rolling train/test.

    Parameters
    ----------
    prices : DataFrame
        Daily close prices.
    tickers : list
        Tickers to trade.
    n_estimators : int
        Number of boosting stages.
    max_depth : int
        Maximum tree depth.
    learning_rate : float
        Shrinkage parameter.
    subsample : float
        Fraction of samples for each tree.
    threshold : float
        Minimum predicted 10-day return to enter.
    max_positions : int
        Maximum concurrent positions.
    train_months : int
        Training window in months.
    rebal_weeks : int
        Rebalance every N weeks.
    name : str
        Strategy label.

    Returns
    -------
    dict with equity curve and metrics.
    """
    p = prices[tickers].copy()
    p = p.dropna()
    daily_ret = p.pct_change().fillna(0)

    train_days = train_months * 21
    rebal_days = rebal_weeks * 5
    lookback = 90
    min_start = max(train_days, lookback + 60)

    if len(p) < min_start + 100:
        return None

    equity = pd.Series(index=p.index, dtype=float)
    equity.iloc[:min_start] = 1.0

    model = None
    scaler = None
    positions = {}
    importances_log = []
    position_size = 0.90 / max(max_positions, 1)

    for i in range(min_start, len(p)):
        days_since_start = i - min_start
        is_rebal = (days_since_start % rebal_days == 0)
        is_train = (days_since_start % (21 * 1) == 0)  # monthly training

        # Train model monthly
        if is_train or model is None:
            all_X, all_y = [], []
            for ticker in tickers:
                if ticker not in p.columns:
                    continue
                if i - lookback < 0:
                    continue
                ticker_prices = p[ticker].iloc[max(0, i-train_days):i]
                if len(ticker_prices) < 60:
                    continue

                df = pd.DataFrame({
                    'close': ticker_prices,
                    'high': ticker_prices,
                    'low': ticker_prices,
                    'volume': pd.Series(1e6, index=ticker_prices.index)
                })
                feats = calculate_features(df)
                # Target: 10-day forward return
                target = ticker_prices.pct_change(10).shift(-10)

                feats['target'] = target
                feats = feats.dropna()
                if len(feats) > 20:
                    all_X.append(feats.drop('target', axis=1))
                    all_y.append(feats['target'])

            if all_X:
                X = pd.concat(all_X, ignore_index=True)
                y = pd.concat(all_y, ignore_index=True)

                scaler = StandardScaler()
                X_scaled = scaler.fit_transform(X)

                model = GradientBoostingRegressor(
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    learning_rate=learning_rate,
                    min_samples_leaf=10,
                    subsample=subsample,
                    random_state=42
                )
                model.fit(X_scaled, y)
                importances_log.append(dict(zip(X.columns, model.feature_importances_)))

        # Rebalance
        if is_rebal and model is not None:
            predictions = {}
            for ticker in tickers:
                if ticker not in p.columns:
                    continue
                if i - lookback < 0:
                    continue
                ticker_prices = p[ticker].iloc[i-lookback:i]
                if len(ticker_prices) < 50:
                    continue

                df = pd.DataFrame({
                    'close': ticker_prices,
                    'high': ticker_prices,
                    'low': ticker_prices,
                    'volume': pd.Series(1e6, index=ticker_prices.index)
                })
                feats = calculate_features(df)
                if len(feats) == 0:
                    continue

                latest = feats.iloc[-1:].values
                latest = scaler.transform(latest)
                pred = model.predict(latest)[0]
                predictions[ticker] = pred

            sorted_preds = sorted(predictions.items(), key=lambda x: x[1], reverse=True)
            selected = [(t, pred) for t, pred in sorted_preds
                        if pred > threshold][:max_positions]

            if selected:
                positions = {t: position_size for t, _ in selected}
            else:
                positions = {}

        # Daily return
        port_ret = 0.0
        for ticker, w in positions.items():
            if ticker in daily_ret.columns:
                port_ret += w * daily_ret.iloc[i][ticker]

        equity.iloc[i] = equity.iloc[i-1] * (1.0 + port_ret)

    # Metrics
    returns = equity.pct_change().dropna()
    n_years = (equity.index[-1] - equity.index[0]).days / 365.25
    final = equity.iloc[-1]
    start_val = equity[equity > 0].iloc[0]
    cagr = (final / start_val) ** (1.0 / n_years) - 1.0
    vol = returns.std() * np.sqrt(252)
    sharpe = (returns.mean() / returns.std()) * np.sqrt(252) if returns.std() > 0 else 0
    drawdown = equity / equity.cummax() - 1.0
    max_dd = drawdown.min()

    avg_importance = {}
    if importances_log:
        for key in importances_log[0]:
            avg_importance[key] = np.mean([d[key] for d in importances_log])

    return {
        'name': name,
        'equity': equity,
        'returns': returns,
        'sharpe': round(sharpe, 3),
        'cagr': round(cagr * 100, 2),
        'max_dd': round(max_dd * 100, 2),
        'volatility': round(vol * 100, 2),
        'feature_importance': avg_importance,
    }


def compare_results(results_list):
    """Print comparison table of multiple backtest results."""
    rows = []
    for r in results_list:
        if r is None:
            continue
        rows.append({
            'Strategy': r['name'],
            'Sharpe': r['sharpe'],
            'CAGR (%)': r['cagr'],
            'MaxDD (%)': r['max_dd'],
            'Vol (%)': r['volatility'],
        })
    df = pd.DataFrame(rows).set_index('Strategy')
    print(df.to_string())
    print()
    best_sharpe = df['Sharpe'].idxmax()
    print(f"  >> Best Sharpe: {best_sharpe} ({df.loc[best_sharpe, 'Sharpe']})")
    return df


def plot_equity_curves(results_list, title='Equity Curves'):
    """Plot equity curves and drawdowns."""
    fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1]})

    ax1 = axes[0]
    for r in results_list:
        if r is None:
            continue
        label = f"{r['name']} (S={r['sharpe']}, DD={r['max_dd']}%)"
        ax1.plot(r['equity'], label=label, linewidth=1.2)
    ax1.set_title(title, fontsize=14)
    ax1.set_ylabel('Portfolio Value')
    ax1.legend(fontsize=8, loc='upper left')
    ax1.grid(True, alpha=0.3)

    ax2 = axes[1]
    for r in results_list:
        if r is None:
            continue
        dd = r['equity'] / r['equity'].cummax() - 1.0
        ax2.fill_between(dd.index, dd.values, alpha=0.3, label=r['name'])
    ax2.set_title('Drawdowns', fontsize=12)
    ax2.set_ylabel('Drawdown')
    ax2.legend(fontsize=8, loc='lower left')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

print("Backtest engine loaded.")

### Benchmark SPY buy-and-hold

In [ ]:
# SPY benchmark
spy_ret = prices[BENCHMARK].pct_change().fillna(0)
spy_equity = (1 + spy_ret).cumprod()
n_years = (prices.index[-1] - prices.index[0]).days / 365.25
spy_sharpe = (spy_ret.mean() / spy_ret.std()) * np.sqrt(252)
spy_cagr = (prices[BENCHMARK].iloc[-1] / prices[BENCHMARK].iloc[0]) ** (1/n_years) - 1
spy_dd = ((spy_equity / spy_equity.cummax()) - 1).min()

benchmark_result = {
    'name': f'{BENCHMARK} Buy-Hold',
    'equity': spy_equity,
    'returns': spy_ret,
    'sharpe': round(spy_sharpe, 3),
    'cagr': round(spy_cagr * 100, 2),
    'max_dd': round(spy_dd * 100, 2),
    'volatility': round(spy_ret.std() * np.sqrt(252) * 100, 2),
}
print(f"Benchmark {BENCHMARK}: Sharpe={benchmark_result['sharpe']}, "
      f"CAGR={benchmark_result['cagr']}%, MaxDD={benchmark_result['max_dd']}%")

## Hypothese 1 : Learning rate

### Rationale

Le learning rate controle la contribution de chaque arbre dans le boosting :
- **0.01** : Convergence tres lente, necessite plus d'estimateurs, mais meilleure generalisation
- **0.03 (actuel)** : Compromis standard pour le gradient boosting
- **0.05** : Apprentissage plus rapide, risque de surajustement modere
- **0.10** : Rapide mais surapprend souvent les donnees financieres bruitees

Un LR trop bas avec peu d'estimateurs sous-exploite le modele. Un LR trop haut
surapprend les fluctuations recentes du marche.

In [ ]:
# H1: Learning rate
lr_values = [0.01, 0.03, 0.05, 0.10]
h1_results = []

for lr in lr_values:
    print(f"Testing learning_rate={lr}...")
    r = backtest_xgb_strategy(
        prices, TICKERS_15,
        n_estimators=100, max_depth=5, learning_rate=lr,
        subsample=0.8, threshold=0.001, max_positions=9,
        rebal_weeks=2, name=f'lr={lr}'
    )
    h1_results.append(r)

h1_results.append(benchmark_result)
print("\n=== H1: Learning Rate ===")
h1_df = compare_results(h1_results)
plot_equity_curves(h1_results, title='H1: Learning Rate')

### Interpretation H1

- Si le Sharpe s'amelior avec un LR plus bas, le modele actuel surapprend.
- Un LR plus eleve qui degrade le Sharpe confirme que les donnees financieres
  sont trop bruitees pour un apprentissage agressif.
- Le CAGR devrait suivre la meme tendance que le Sharpe.

## Hypothese 2 : Nombre d'estimateurs

### Rationale

Le nombre de stages de boosting affecte la capacite du modele :
- **50** : Modele trop simple, sous-apprentissage probable
- **100 (actuel)** : Compromis standard
- **200** : Capacite accrue, mais risque d'overfitting sur les donnees d'entrainement
- **300** : Rendements decroissants attendus, cout computationnel eleve

Le gradient boosting est plus sensible au surapprentissage que le Random Forest
car chaque arbre corrige les erreurs du precedent.

In [ ]:
# H2: N estimators
n_est_values = [50, 100, 200, 300]
h2_results = []

for n_est in n_est_values:
    print(f"Testing n_estimators={n_est}...")
    r = backtest_xgb_strategy(
        prices, TICKERS_15,
        n_estimators=n_est, max_depth=5, learning_rate=0.03,
        subsample=0.8, threshold=0.001, max_positions=9,
        rebal_weeks=2, name=f'n_est={n_est}'
    )
    h2_results.append(r)

h2_results.append(benchmark_result)
print("\n=== H2: N Estimators ===")
h2_df = compare_results(h2_results)
plot_equity_curves(h2_results, title='H2: N Estimators')

### Interpretation H2

- Le gradient boosting montre souvent un plateau de performance au-dela de 200 estimateurs.
- Si 50 estimateurs performent presque aussi bien que 100, le modele n'a pas besoin
  d'autant de complexity et l'entrainement peut etre accelere.
- Le MaxDD ne devrait pas beaucoup changer avec le nombre d'estimateurs.

## Hypothese 3 : Seuil de prediction

### Rationale

Le seuil de rendement predit (threshold) filtre les signaux faibles :
- **0.0** : Aucun filtre, prend toutes les predictions positives
- **0.001 (actuel)** : Filtre minimal, elimine les predictions negatives ou nulles
- **0.005** : Seuil modere, exige un rendement 10j predit > 0.5%
- **0.01** : Seuil strict, rendement 10j predit > 1%

Le seuil actuel (0.001) est tres permissif. Un seuil plus eleve pourrait
ameliorer la selectivite au prix de moins de positions.

In [ ]:
# H3: Prediction threshold
threshold_values = [0.0, 0.001, 0.005, 0.01]
h3_results = []

for thr in threshold_values:
    print(f"Testing threshold={thr}...")
    r = backtest_xgb_strategy(
        prices, TICKERS_15,
        n_estimators=100, max_depth=5, learning_rate=0.03,
        subsample=0.8, threshold=thr, max_positions=9,
        rebal_weeks=2, name=f'threshold={thr}'
    )
    h3_results.append(r)

h3_results.append(benchmark_result)
print("\n=== H3: Prediction Threshold ===")
h3_df = compare_results(h3_results)
plot_equity_curves(h3_results, title='H3: Prediction Threshold')

### Interpretation H3

- Si threshold=0.0 performe bien, la direction de la prediction (pas la magnitude) est le signal principal.
- Un seuil plus strict qui ameliore le Sharpe indique que les predictions fortes sont plus fiables.
- Le nombre de positions actives devrait diminuer avec le seuil, augmentant le temps en cash.

## Hypothese 4 : Nombre maximum de positions

### Rationale

Le nombre de positions affecte la concentration du portefeuille :
- **3 positions** : Hautement concentre, potentiel de gain eleve mais risque max
- **5** : Compromis concentration/diversification
- **7** : Bonne diversification, poids ~13% par position
- **9 (actuel)** : Tres diversifie, poids ~10% par position
- **12** : Sur-diversification possible, dilution du signal

Avec un modele de regression qui predit les rendements, plus de positions
permet de diversifier le risque de prediction individuelle.

In [ ]:
# H4: Max positions
pos_values = [3, 5, 7, 9, 12]
h4_results = []

for max_pos in pos_values:
    print(f"Testing max_positions={max_pos}...")
    r = backtest_xgb_strategy(
        prices, TICKERS_15,
        n_estimators=100, max_depth=5, learning_rate=0.03,
        subsample=0.8, threshold=0.001, max_positions=max_pos,
        rebal_weeks=2, name=f'max_pos={max_pos}'
    )
    h4_results.append(r)

h4_results.append(benchmark_result)
print("\n=== H4: Max Positions ===")
h4_df = compare_results(h4_results)
plot_equity_curves(h4_results, title='H4: Max Positions')

### Interpretation H4

- Un nombre reduit de positions (3-5) amplifie le signal du modele mais augmente le MaxDD.
- Un nombre eleve (12) dilue le signal mais reduit le risque individuel.
- Le sweet spot depend de la fiabilite du modele : si les top predictions sont fiables,
  moins de positions est preferable.

## Hypothese 5 : Subsample ratio

### Rationale

Le subsampling introduit de l'aleatoire dans le boosting (stochastic gradient boosting) :
- **0.6** : Forte stochasticite, reduit la variance mais augmente le biais
- **0.7** : Bon compromis entre stochasticite et stabilite
- **0.8 (actuel)** : Peu de stochasticite, modele stable
- **1.0** : Pas de stochasticite, gradient boosting deterministe

Le subsampling agit comme une forme de regularisation et reduit le surapprentissage
sans sacrifier la capacite du modele.

In [ ]:
# H5: Subsample ratio
subsample_values = [0.6, 0.7, 0.8, 1.0]
h5_results = []

for ss in subsample_values:
    print(f"Testing subsample={ss}...")
    r = backtest_xgb_strategy(
        prices, TICKERS_15,
        n_estimators=100, max_depth=5, learning_rate=0.03,
        subsample=ss, threshold=0.001, max_positions=9,
        rebal_weeks=2, name=f'subsample={ss}'
    )
    h5_results.append(r)

h5_results.append(benchmark_result)
print("\n=== H5: Subsample Ratio ===")
h5_df = compare_results(h5_results)
plot_equity_curves(h5_results, title='H5: Subsample Ratio')

### Interpretation H5

- Le stochastic gradient boosting (subsample < 1.0) est generalement preferable
  pour les donnees financieres car il reduit le surapprentissage.
- Si subsample=1.0 performe aussi bien, le surapprentissage n'est pas un probleme majeur.
- Un subsample trop bas (0.6) peut degrader les performances si le dataset est deja limite.

## Synthese : Analyse de l'importance des features

In [ ]:
# Feature importance from the baseline run
baseline = backtest_xgb_strategy(
    prices, TICKERS_15,
    n_estimators=100, max_depth=5, learning_rate=0.03,
    subsample=0.8, threshold=0.001, max_positions=9,
    rebal_weeks=2, name='Baseline'
)

if baseline and baseline['feature_importance']:
    imp = pd.Series(baseline['feature_importance']).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(10, 8))
    imp.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Feature Importance (GradientBoosting, moyenne sur tous les entrainements)', fontsize=13)
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()

    print("\nFeature importance ranking:")
    for feat, val in imp.sort_values(ascending=False).items():
        print(f"  {feat:20s}: {val:.4f}")
else:
    print("Baseline backtest did not produce feature importance.")

## Conclusions et recommandations

### Configuration recommandee pour main.py

Basee sur les resultats des hypotheses, remplir le tableau ci-dessous apres execution :

| Parametre | Valeur actuelle | Valeur recommandee | Justification |
|-----------|----------------|--------------------|--------------|
| learning_rate | 0.03 | A determiner | Resultat H1 |
| n_estimators | 100 | A determiner | Resultat H2 |
| threshold | 0.001 | A determiner | Resultat H3 |
| max_positions | 9 | A determiner | Resultat H4 |
| subsample | 0.8 | A determiner | Resultat H5 |

### Limites de cette analyse

- **Pas de frais de transaction** : Le backtest standalone ne modelise pas les commissions QC
- **Donnees ajustees uniquement** : yfinance fournit des prix ajustes, pas les prix d'execution
- **High/Low approximes** : Les features utilisent close comme approximation de high/low
- **Look-ahead risque** : Les meilleurs parametres sont choisis sur l'echantillon complet
- **Periode unique** : 2015-2026, un walk-forward sur plusieurs sous-periodes serait plus robuste
- **GradientBoostingRegressor vs XGBRegressor** : On utilise sklearn, pas la librairie xgboost native.
  Les performances peuvent differer legerement.

### Prochaines etapes

1. Implementer les parametres recommandes dans `main.py`
2. Backtester sur QC Cloud pour valider avec les couts reels
3. Comparer le Sharpe standalone vs Sharpe QC pour estimer l'impact des frais
4. Considerer un walk-forward test sur 3 sous-periodes (2015-2018, 2018-2022, 2022-2026)